<a href="https://colab.research.google.com/github/Jeevith252/ML_BASIC_PROJECT/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
text = "NLP ,, Session is... going on!!!"
clean = re.sub(r'[^a-zA-Z]' , '' ,text)
print(clean)

NLPSessionisgoingon


In [ ]:
text = "The climate is beautiful"
tokens = text.split()
print(tokens)

['The', 'climate', 'is', 'beautiful']


In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
text = "The climate is beautiful"
tokens = word_tokenize(text)
filtered = [
    word for word in tokens
    if word.lower() not in stopwords.words("english")
]
print(filtered)

['climate', 'beautiful']


In [ ]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
words = ["played" , "reading"]
for word in words:
  print(stemmer.stem(word))

play
read


In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
print(lemmatizer.lemmatize("running", pos="v"))

[nltk_data] Downloading package wordnet to /root/nltk_data...


run


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
text = ["The climate is beautiful" , "It is raining"]
vector = CountVectorizer()

In [ ]:
x = vector.fit_transform(text)
print(x.toarray())

[[1 1 1 0 0 1]
 [0 0 1 1 1 0]]


In [ ]:
print(vector.get_feature_names_out())

['beautiful' 'climate' 'is' 'it' 'raining' 'the']


In [ ]:
pip install gensim --q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 26.4 MB/s eta 0:00:00


In [ ]:
from gensim.models import Word2Vec



sentences = ["climate", "is", "beautiful"],["it", "is", "raining"]
model = Word2Vec(sentences,vector_size=20,window=2,min_count=1,workers=4)

print(model.wv["climate"])

[-0.00428278  0.01413282  0.02700714  0.03526328 -0.02851561  0.0092941
  0.03044432 -0.02399025 -0.0155363   0.03398815  0.00815738  0.00094959
  0.01736819  0.00108889  0.04809413  0.02530302 -0.04458695 -0.0352078
  0.00450728  0.03196267]


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

text = ["I love this phone","I hate this phone's feature"]

labels = [1, 0]
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(text)
model = LogisticRegression()
model.fit(X, labels)

new_text = ["i like this phone"]
new_x = vectorizer.transform(new_text)
predictions = model.predict(new_x)
print(predictions)

[1]


# **IMDB Project**

In [8]:
!pip install -q nltk spacy xgboost
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 26.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [9]:
import pandas as pd
import re,string,nltk,spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score , classification_report ,confusion_matrix
nltk.download("punkt_tab")
nltk.download("stopwords")
nlp = spacy.load("en_core_web_sm")



[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


**Loading the dataset**

In [10]:
df = pd.read_csv("/content/IMDB Dataset.csv")

In [11]:
df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [13]:
df.describe()

,review,sentiment
count,50000,50000
unique,49582,2
top,Loved today's show!!! It was a variety and not...,positive
freq,5,25000


In [14]:
df.isnull().sum()

,0
review,0
sentiment,0


In [15]:
df.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
49995,False
49996,False
49997,False
49998,False


In [16]:
stop = stopwords.words("english")

def preprocess(text):
  text = text.lower()

  text = re.sub(r"<.*?>"," ",text)
  text = re.sub(r'http\S+|www|S+' , ' ' ,text)
  text = re.sub(r'[^a-z\s]',' ',text)

  tokens = word_tokenize(text)
  tokens = [t for t in tokens if t not in stop]

  doc = nlp(" ".join(tokens))

  lemmas = [tok.lemma_ for tok in doc]
  return ' '.join(lemmas)

In [17]:
df["clean_review"] = df["review"].apply(preprocess)

df[["review" , "clean_review"]].head()

,review,clean_review
0,One of the other reviewers has mentioned that ...,one reviewer mention watch oz episode hook rig...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,think wonderful way spend time hot summer week...
3,Basically there's a family where a little boy ...,basically family little boy jake think zombie ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stunnin...


In [18]:
df['label'] = df['sentiment'].map({'negative': 0 , "positive" : 1})

X_train , X_test ,y_train , y_test = train_test_split(df["clean_review"] , df["label"] ,test_size = 0.2 , random_state =42 ,stratify = df["label"])

In [19]:
tfidf = TfidfVectorizer(max_features = 5000)
X_train_t = tfidf.fit_transform(X_train)
X_test_t = tfidf.transform(X_test)
print(X_train_t.shape)

(40000, 5000)


In [21]:
lr = LogisticRegression(max_iter = 1000)
lr.fit(X_train_t , y_train)
pred_lr = lr.predict(X_test_t)
print("LR Accuracy",accuracy_score(y_test , pred_lr))
print(confusion_matrix(y_test,pred_lr))
print(classification_report(y_test,pred_lr))


NameError: name 'LogisticRegression' is not defined

In [22]:
xgb = XGBClassifier(n_estimators = 100 ,max_depth = 6,learning_rate = 0.1, eval_metric = "logloss", random_state =42)
xgb.fit(X_train_t,y_train)
xgb_pred = xgb.predict(X_test_t)
print("XGB accuracy :",accuracy_score(y_test,xgb_pred))
print(confusion_matrix(y_test,xgb_pred))
print(classification_report(y_test,xgb_pred))

XGB accuracy : 0.831
[[4005  995]
 [ 695 4305]]
              precision    recall  f1-score   support

           0       0.85      0.80      0.83      5000
           1       0.81      0.86      0.84      5000

    accuracy                           0.83     10000
   macro avg       0.83      0.83      0.83     10000
weighted avg       0.83      0.83      0.83     10000



In [ ]:
results = pd.DataFrame({
    "Model" : ["Logistic Regression" , "XGBoost"],
    "Accuracy" : [accuracy_score(y_test , pred_lr),
                  accuracy_score(y_test,xgb_pred)]
})

print(results)

In [ ]:
def predict_review(review,model):
  clean = preprocess(review)
  vec = tfidf.transform([clean])
  pred = model.predict(vec)[0]

  return "Positive" if pred == 1 else "Negative"

print(predict_review("This movie was aboslutely fantastic." ,lr))
print(predict_review("Worst movie I have ever watched." , xgb))

##Hugging face transformers



In [ ]:
!pip install evaluate

In [25]:

import pandas as pd
from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np
from sklearn.model_selection import train_test_split

In [26]:
df=pd.read_csv("/content/IMDB Dataset.csv")
df['label']=df['sentiment'].map({'negative':0,'positive':1})
df=df[['review','label']]
train_df,test_df=train_test_split(df,test_size=0.2,random_state=42,stratify=df['label'])
train_ds=Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds=Dataset.from_pandas(test_df.reset_index(drop=True))
train_ds,test_ds

(Dataset({
     features: ['review', 'label'],
     num_rows: 40000
 }),
 Dataset({
     features: ['review', 'label'],
     num_rows: 10000
 }))

In [27]:
tokenizer=DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
def tokenize(batch):
  return tokenizer(batch['review'],padding='max_length',truncation=True,max_length=256)

train_ds=train_ds.map(tokenize,batched=True)
test_ds=test_ds.map(tokenize,batched=True)
print("\nColumns after tokenization:")
print(train_ds.column_names)
print("\nFirst tokenized examples:")
print(train_ds[0])
cols=["input_ids","attention_mask","label"]
train_ds.set_format(type='torch',columns=cols)
test_ds.set_format(type='torch',columns=cols)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]


Columns after tokenization:
['review', 'label', 'input_ids', 'attention_mask']

First tokenized examples:
{'review': 'I caught this little gem totally by accident back in 1980 or \'81. I was at a revival theatre to see two old silly sci-fi movies. The theatre was packed full and (with no warning) they showed a bunch of sci-fi short spoofs (to get us in the mood). Most were somewhat amusing but THIS came on and, within seconds, the audience was in hysterics! The biggest laugh came when they showed "Princess Laia" having huge cinnamon buns instead of hair on her head. She looks at the camera, gives a grim smile and nods. That made it even funnier! You gotta see "Chewabacca" played by what looks like a Muppet! It was extremely silly and stupid...but I couldn\'t stop laughing. Most of the dialogue was drowned out because of all the laughter. Also if you know "Star Wars" pretty well it\'s even funnier--they deliberately poke fun at some of the dialogue. This REALLY works with an audience! 

In [28]:

model=DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased',num_labels=2)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [29]:
accuracy=evaluate.load('accuracy')
precision=evaluate.load('precision')
recall=evaluate.load('recall')
f1=evaluate.load('f1')

def compute_metrics(eval_pred):
  logits , labels = eval_pred
  preds=np.argmax(logits,axis=-1)
  return {
      'accuracy':accuracy.compute(predictions=preds,references=labels)['accuracy'],
      'precision':precision.compute(predictions=preds,references=labels)['precision'],
      'recall':recall.compute(predictions=preds,references=labels)['recall'],
      'f1':f1.compute(predictions=preds,references=labels)['f1']
  }

In [30]:
training_args=TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100
)

In [31]:
trainer=Trainer(
    model = model,
    args = training_args,
    train_dataset = train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [32]:
!pip install torchvision==0.1.6

Reason for being yanked: 0.1.6 is past it's support date and confuses users on unsupported platforms
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.26.0+cpu
    Uninstalling torchvision-0.26.0+cpu:
      Successfully uninstalled torchvision-0.26.0+cpu
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.7 requires torchvision>=0.11, but you have torchvision 0.1.6 which is incompatible.


In [ ]:
print(trainer.train())
print(trainer.evaluate())

In [ ]:
from transformers import pipeline

classifier = pipeline('sentiment-analysis', model = 'distilbert_imdb', tokenizer = 'distilbert_imdb')

print(classifier('This movie was absolutely fantastic.'))
print(classifier("Worst movie ever made."))